# 05 — Dashboard data exports

Assemble the **dashboard-ready** datasets and the summary tables the app needs,
and write them to `data/processed/` and `outputs/tables/`.

> **Governance reminder.** This dashboard produces a *triage* signal for human policy review. It does **not** determine social-assistance need, eligibility, or benefits. Claude outputs are claims, not facts. Missing or suppressed values are flagged, never invented. Any policy interpretation is reviewed by the AI Council before it is treated as decision-ready.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # import project modules
import utils
print("Project root:", utils.ROOT)
print("Raw data:    ", utils.RAW)
print("Processed:   ", utils.PROCESSED)
print("Outputs:     ", utils.OUTPUTS)
utils.ensure_dirs()

In [ ]:
import pandas as pd, numpy as np
panel = pd.read_csv(utils.PROCESSED / "policy_triage_panel.csv", low_memory=False)
print(f"panel: {len(panel):,} rows · {panel['ref_date'].min()} → {panel['ref_date'].max()}")

## Optional: merge annual context (income / low income / population)

Annual sources join to the monthly spine **by `year`** and stay clearly labelled annual context. The same annual figure repeats across that year's months by design — the dashboard must say so. Run after notebooks 01-02 have produced the `*_clean.csv`.

In [ ]:
def merge_annual(panel, fname, geo_col="geo", value_col=None, new_col=None):
    p = utils.PROCESSED / fname
    if not p.exists():
        print(f"skip {fname} (not downloaded yet)"); return panel
    ctx = pd.read_csv(p, low_memory=False)
    # ... select the headline slice, group to geo×year, rename to new_col, merge how='left'
    print(f"would merge {fname} by geo+year as labelled annual context")
    return panel

# Example (uncomment when files exist):
# panel = merge_annual(panel, "income_clean.csv", new_col="income_value")
# panel = merge_annual(panel, "demographics_clean.csv", new_col="population")

## Executive summary table (latest month, by province)

In [ ]:
latest = panel.sort_values("ref_date").groupby("geo").tail(1).copy()
summary = latest[["geo","ref_date","unemployment_rate","youth_unemployment_rate",
                  "employment_rate","participation_rate","unemp_change",
                  "policy_review_priority_score","score_confidence","confidence_flag"]] \
            .sort_values("policy_review_priority_score", ascending=False)
summary.to_csv(utils.TABLES / "latest_month_triage_ranking.csv", index=False)
summary

## Trend export (long, dashboard-friendly)

In [ ]:
trend = panel[["geo","ref_date","year","month","unemployment_rate","employment_rate",
               "participation_rate","youth_unemployment_rate",
               "policy_review_priority_score","confidence_flag","missing_value_flag"]]
trend.to_csv(utils.TABLES / "triage_trend_long.csv", index=False)
print("wrote outputs/tables/triage_trend_long.csv", trend.shape)

## Final dashboard dataset

`data/processed/policy_triage_panel.csv` is the main input the dashboard reads (`dashboard/app.py`). It includes the score, its explanation, confidence, and missing-value flags. Re-run notebooks 01-04 to refresh it, then this notebook to re-export.

In [ ]:
print("Dashboard reads:", utils.PROCESSED / "policy_triage_panel.csv")
print("Columns:", list(panel.columns))